# TEMPO Mid-Atlantic L3 Download

Download TEMPO NO2, Ozone, and Formaldehyde L3 products (V003 and V004) for a mid-Atlantic slice covering **May 28 – June 5, 2025**.

| | |
|---|---|
| **Products** | NO2 (`TEMPO_NO2_L3`), Total Column O3 (`TEMPO_O3TOT_L3`), Formaldehyde (`TEMPO_HCHO_L3`) |
| **Versions** | V003, V004 |
| **Region** | 34–43 °N, 77–69 °W (North Carolina → Boston) |
| **Dates** | 2025-05-28 → 2025-06-05 |

In [1]:
from pathlib import Path

import earthaccess
import xarray as xr
import pandas as pd

from atmoz.data_access.NASA import EarthData

/home/magnolia/research/STAQS_Analysis/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Geographic bounding box: (W, S, E, N)
BBOX = (-77.0, 34.0, -69.0, 43.0)

DATE_START = "2025-05-28"
DATE_END   = "2025-06-05"

VERSIONS = ["003", "004"]

PRODUCTS = {
    "NO2":  "TEMPO_NO2_L3",
    "O3":   "TEMPO_O3PROF_L3",
    "HCHO": "TEMPO_HCHO_L3",
}

## 1. Discover Available TEMPO L3 Products

Query NASA CMR to confirm short names and available versions before downloading.

In [4]:
EarthData.store_credentials(username="mroots", password="OhAwesomeNASAdata12$")

In [5]:
nasa = EarthData()

all_tempo = nasa.get_short_names(keyword="TEMPO_*")
l3 = all_tempo[all_tempo["ShortName"].str.contains("L3")][["ShortName", "EntryTitle"]].reset_index(drop=True)
l3

,ShortName,EntryTitle
0,TEMPO_NO2_L3,TEMPO gridded NO2 tropospheric and stratospher...
1,TEMPO_NO2_L3_NRT,TEMPO gridded NO2 tropospheric and stratospher...
2,TEMPO_O3TOT_L3,TEMPO gridded ozone total column V04 (PROVISIO...
3,TEMPO_O3PROF_L3,TEMPO gridded ozone profile V04 (BETA)
4,TEMPO_CLDO4_L3,TEMPO gridded cloud fraction and pressure (O2-...
5,TEMPO_HCHO_L3,TEMPO gridded formaldehyde total column V04 (...
6,TEMPO_NO2_L3,TEMPO gridded NO2 tropospheric and stratospher...
7,TEMPO_CLDO4_L3_NRT,TEMPO gridded cloud fraction and pressure (O2-...
8,TEMPO_HCHO_L3_NRT,TEMPO gridded formaldehyde total column V02 (N...
9,TEMPO_HCHO_L3,TEMPO gridded formaldehyde total column V03 (P...


In [ ]:
all_tempo

,ShortName,EntryTitle,Abstract,Response
0,TEMPO_O3TOT_L2,TEMPO ozone total column V04 (PROVISIONAL),Total ozone Level 2 (PROVISIONAL) files provi...,"{'DataLanguage': 'English', 'AncillaryKeywords..."
1,TEMPO_NO2_L2,TEMPO NO2 tropospheric and stratospheric colum...,Nitrogen dioxide Level 2 (PROVISIONAL) files p...,"{'DataLanguage': 'English', 'AncillaryKeywords..."
2,TEMPO_CLDO4_L2,TEMPO cloud pressure and fraction (O2-O2 dimer...,O2-O2 cloud Level 2 (PROVISIONAL) files provid...,"{'DataLanguage': 'English', 'AncillaryKeywords..."
3,TEMPO_O3PROF_L2,TEMPO ozone profile V04 (BETA),Ozone profile Level 2 (BETA) files provide ozo...,"{'DataLanguage': 'English', 'AncillaryKeywords..."
4,TEMPO_HCHO_L2,TEMPO formaldehyde total column V04 (PROVISION...,Formaldehyde Level 2 (PROVISIONAL) files provi...,"{'DataLanguage': 'English', 'AncillaryKeywords..."
5,TEMPO_NO2_L2_NRT,TEMPO NO2 tropospheric and stratospheric colum...,Nitrogen dioxide Level 2 (PROVISIONAL) files p...,"{'DataLanguage': 'English', 'CollectionCitatio..."
6,TEMPO_CLDO4_L2_NRT,TEMPO cloud pressure and fraction (O2-O2 dimer...,O2-O2 cloud Level 2 (PROVISIONAL) files provid...,"{'DataLanguage': 'English', 'CollectionCitatio..."
7,TEMPO_NO2_L3,TEMPO gridded NO2 tropospheric and stratospher...,Nitrogen dioxide Level 3 (PROVISIONAL) files p...,"{'DataLanguage': 'English', 'AncillaryKeywords..."
8,TEMPO_NO2_L3_NRT,TEMPO gridded NO2 tropospheric and stratospher...,Nitrogen dioxide Level 3 (PROVISIONAL) files p...,"{'DataLanguage': 'English', 'CollectionCitatio..."
9,TEMPO_O3TOT_L3,TEMPO gridded ozone total column V04 (PROVISIO...,Total ozone Level 3 (PROVISIONAL) files provid...,"{'DataLanguage': 'English', 'AncillaryKeywords..."


## 2. Search → Open Lazily → Slice

For each product × version: find granules in CMR, open them as file-like objects via `earthaccess.open()`, combine into a single xarray Dataset with `open_mfdataset`, then slice to the bounding box.  
Nothing is written to disk — only the mid-Atlantic pixels are read into memory when you access them.

In [6]:
VERSIONS = ["V03"]

PRODUCTS = {
    "NO2":  "TEMPO_NO2_L3",
    "O3":   "TEMPO_O3TOT_L3",
    "HCHO": "TEMPO_HCHO_L3",
}

for label, short_name in PRODUCTS.items():
    for version in VERSIONS: 
        granules = nasa.search_data(
            short_name=short_name,
            version=version,
            temporal=(DATE_START, DATE_END),
            bounding_box=BBOX,
            )
        
        print(f"{short_name} ({version}): {len(granules)} granule(s) found — opening lazily ...")



TEMPO_NO2_L3 (V03): 119 granule(s) found — opening lazily ...
TEMPO_O3TOT_L3 (V03): 119 granule(s) found — opening lazily ...
TEMPO_HCHO_L3 (V03): 119 granule(s) found — opening lazily ...


In [7]:
nasa = EarthData()
datasets = {}

for label, short_name in PRODUCTS.items():
    for version in VERSIONS:
        key = f"{label}_{version}"
        print(f"\n[{key}] Searching {short_name} {version} ...")

        granules = nasa.search_data(
            short_name=short_name,
            version=version,
            temporal=(DATE_START, DATE_END),
            bounding_box=BBOX,
        )
        print(f"  {len(granules)} granule(s) found — opening lazily ...")

        if not granules:
            print("  No granules found, skipping.")
            continue

        fobjs = earthaccess.open(granules)
        ds = xr.open_mfdataset(fobjs, engine="h5netcdf", combine="by_coords")

        # Slice to mid-Atlantic bounding box (lat/lon coord names may vary by product version)
        lat_dim = "lat" if "lat" in ds.coords else "latitude"
        lon_dim = "lon" if "lon" in ds.coords else "longitude"
        ds_slice = ds.sel(
            {lat_dim: slice(BBOX[1], BBOX[3]),
             lon_dim: slice(BBOX[0], BBOX[2])}
        )

        datasets[key] = ds_slice
        print(f"  Sliced dims: {dict(ds_slice.dims)}")

print("\nAll datasets opened.")


[NO2_V03] Searching TEMPO_NO2_L3 V03 ...
  119 granule(s) found — opening lazily ...


QUEUEING TASKS | : 100%|██████████| 119/119 [00:00<00:00, 18089.38it/s]
PROCESSING TASKS | : 100%|██████████| 119/119 [00:26<00:00,  4.51it/s]
COLLECTING RESULTS | : 100%|██████████| 119/119 [00:00<00:00, 295163.91it/s]


KeyboardInterrupt: 

## 3. Dataset Summary

In [ ]:
rows = []
for key, ds in datasets.items():
    label, ver = key.split("_V")
    time_vals = ds["time"].values if "time" in ds else []
    rows.append({
        "Key":        key,
        "Short Name": PRODUCTS[label],
        "Version":    ver,
        "Dims":       dict(ds.dims),
        "Variables":  list(ds.data_vars),
        "Time steps": len(time_vals),
        "Time range": f"{str(time_vals[0])[:10]} → {str(time_vals[-1])[:10]}" if len(time_vals) else "N/A",
    })

pd.DataFrame(rows)